Load all data

In [1]:
import pandas as pd
import numpy as np

BASE = "C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk/data"

cities = {
    'MY1':  'London',
    'BIRR': 'Birmingham',
    'MAN3': 'Manchester',
    'NEWC': 'Newcastle',
    'CARD': 'Cardiff',
}

dfs = []
for site_id, name in cities.items():
    df_train = pd.read_csv(f"{BASE}/{name}/merged/{site_id}_merged.csv",
                           parse_dates=['datetime'])
    df_test  = pd.read_csv(f"{BASE}/test/merged/{site_id}_merged_2025.csv",
                           parse_dates=['datetime'])
    dfs.append(df_train)
    dfs.append(df_test)

data = pd.concat(dfs, ignore_index=True)
data = data.sort_values(['city','datetime']).reset_index(drop=True)
data = data.drop_duplicates(subset=['city','datetime'])

print(f"Total rows: {len(data):,}")
print(f"Date range: {data['datetime'].min()} → {data['datetime'].max()}")
print(data.groupby('city').size())

Total rows: 219,120
Date range: 2021-01-01 01:00:00 → 2026-01-01 00:00:00
city
BIRR    43824
CARD    43824
MAN3    43824
MY1     43824
NEWC    43824
dtype: int64


Clip negatives and impute

In [2]:
# Clip negatives
for col in ['NO2','PM2.5','PM10','O3','SO2']:
    data[col] = data[col].clip(lower=0)

# Impute per city
def impute_city(df):
    df = df.copy().sort_values('datetime')
    cols = ['NO2','PM2.5','PM10','O3','temp','humidity','wind_speed','pressure']
    for col in cols:
        df[col] = df[col].ffill(limit=3)
        df[col] = df[col].fillna(df[col].median())
    df['SO2'] = df['SO2'].fillna(0)
    return df

data = data.groupby('city', group_keys=False).apply(impute_city)
data = data.sort_values(['city','datetime']).reset_index(drop=True)

print("Missing after imputation:")
print(data[['NO2','PM2.5','PM10','O3','SO2','temp','humidity','wind_speed','pressure']].isna().sum())

Missing after imputation:
NO2           0
PM2.5         0
PM10          0
O3            0
SO2           0
temp          0
humidity      0
wind_speed    0
pressure      0
dtype: int64


C:\Users\nikes\AppData\Local\Temp\ipykernel_17192\3441894000.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('city', group_keys=False).apply(impute_city)


Build lag features per city

In [3]:
POLL_COLS = ['NO2','PM2.5','PM10','O3','SO2']
WTHR_COLS = ['temp','humidity','wind_speed','pressure']
ALL_COLS  = POLL_COLS + WTHR_COLS

LAG_HOURS    = [1, 2, 3, 6, 12, 24, 48]
ROLLING_WINS = [6, 12, 24, 48, 72]

def build_lags(df):
    df = df.copy().sort_values('datetime').reset_index(drop=True)
    new_cols = {}

    for col in ALL_COLS:
        # Lag features
        for lag in LAG_HOURS:
            new_cols[f'{col}_lag{lag}h'] = df[col].shift(lag)
        # Rolling means
        for win in ROLLING_WINS:
            new_cols[f'{col}_mean{win}h'] = df[col].shift(1).rolling(win).mean()
        # Rolling max
        for win in [6, 12]:
            new_cols[f'{col}_max{win}h'] = df[col].shift(1).rolling(win).max()

    # Trend features — is pollution rising or falling?
    for col in POLL_COLS:
        new_cols[f'{col}_trend_3h']  = df[col].shift(1) - df[col].shift(4)
        new_cols[f'{col}_trend_6h']  = df[col].shift(1) - df[col].shift(7)
        new_cols[f'{col}_vs_24h']    = df[col] - df[col].shift(24)

    # Rate of change for key pollutants
    for col in ['PM10','PM2.5','NO2']:
        new_cols[f'{col}_roc_3h'] = (
            (df[col] - df[col].shift(3)) / (df[col].shift(3) + 1)
        )

    lag_df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    return lag_df

print("Building extended lag features per city...")
data = data.groupby('city', group_keys=False).apply(build_lags)
data = data.sort_values(['city','datetime']).reset_index(drop=True)

lag_cols = [c for c in data.columns if any(x in c for x in 
            ['lag','mean','max','trend','vs_24h','roc'])]
print(f"Lag feature columns created: {len(lag_cols)}")
print(f"Total columns now: {len(data.columns)}")

Building extended lag features per city...


C:\Users\nikes\AppData\Local\Temp\ipykernel_17192\3792742723.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('city', group_keys=False).apply(build_lags)


Lag feature columns created: 144
Total columns now: 155


Add time features

In [4]:
data['hour']        = data['datetime'].dt.hour
data['day_of_week'] = data['datetime'].dt.dayofweek
data['month']       = data['datetime'].dt.month
data['year']        = data['datetime'].dt.year
data['season']      = data['month'].apply(
    lambda m: 0 if m in [12,1,2] else 1 if m in [3,4,5] else 2 if m in [6,7,8] else 3
)
data['is_weekend'] = (data['day_of_week'] >= 5).astype(int)

city_map = {'MY1':0,'BIRR':1,'MAN3':2,'NEWC':3,'CARD':4}
data['city_code'] = data['city'].map(city_map)

print("Time features added.")
print(data[['datetime','hour','month','season','is_weekend','city_code']].head(3))

Time features added.
             datetime  hour  month  season  is_weekend  city_code
0 2021-01-01 01:00:00     1      1       0           0          1
1 2021-01-01 02:00:00     2      1       0           0          1
2 2021-01-01 03:00:00     3      1       0           0          1


Compute DAQI and shift target 24 hours forward

In [5]:
def compute_daqi(row):
    def sub_index(val, bands):
        if pd.isna(val): return 1
        for i, t in enumerate(bands):
            if val <= t: return i + 1
        return 10

    return max(
        sub_index(row['NO2'],   [67,134,200,267,334,400,467,534,600]),
        sub_index(row['PM2.5'], [11,23,35,41,47,53,58,64,70]),
        sub_index(row['PM10'],  [16,33,50,58,66,75,83,91,100]),
        sub_index(row['O3'],    [33,66,100,120,140,160,187,213,240]),
        sub_index(row['SO2'],   [88,177,266,354,443,532,710,887,1064]),
    )

print("Computing DAQI scores...")
data['DAQI_score'] = data.apply(compute_daqi, axis=1)

def daqi_band(s):
    if s <= 3: return 0
    if s <= 6: return 1
    if s <= 9: return 2
    return 3

data['DAQI_now'] = data['DAQI_score'].apply(daqi_band)

# Generate BOTH 6h and 12h targets in one pass
def shift_targets(df):
    df = df.copy().sort_values('datetime')
    df['DAQI_6h']  = df['DAQI_now'].shift(-6)
    df['DAQI_12h'] = df['DAQI_now'].shift(-12)
    return df

data = data.groupby('city', group_keys=False).apply(shift_targets)
data = data.sort_values(['city','datetime']).reset_index(drop=True)

# Drop rows where either target is missing
data = data.dropna(subset=['DAQI_6h','DAQI_12h'])
data['DAQI_6h']  = data['DAQI_6h'].astype(int).replace({3: 2})
data['DAQI_12h'] = data['DAQI_12h'].astype(int).replace({3: 2})

print(f"Rows after dropping no-target rows: {len(data):,}")

for horizon, col in [('6h','DAQI_6h'),('12h','DAQI_12h')]:
    print(f"\n{horizon} forecast target distribution (3-class):")
    for i, label in enumerate(['Low','Moderate','Dangerous']):
        n = (data[col]==i).sum()
        print(f"  {i} {label:10s}: {n:6,}  ({n/len(data)*100:.1f}%)")

Computing DAQI scores...


C:\Users\nikes\AppData\Local\Temp\ipykernel_17192\3127410006.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('city', group_keys=False).apply(shift_targets)


Rows after dropping no-target rows: 219,060

6h forecast target distribution (3-class):
  0 Low       : 214,372  (97.9%)
  1 Moderate  :  4,158  (1.9%)
  2 Dangerous :    530  (0.2%)

12h forecast target distribution (3-class):
  0 Low       : 214,373  (97.9%)
  1 Moderate  :  4,157  (1.9%)
  2 Dangerous :    530  (0.2%)


Drop rows with NaN lag features and define feature set

In [6]:
data = data.dropna(subset=lag_cols)
data = data.reset_index(drop=True)

print(f"Rows after dropping NaN lag rows: {len(data):,}")

TIME_FEATS = ['city_code','hour','day_of_week','month','season','is_weekend']
FEATURES   = TIME_FEATS + lag_cols

print(f"Total features: {len(FEATURES)}")
print(f"  Time features:  {len(TIME_FEATS)}")
print(f"  Lag features:   {len(lag_cols)}")
print(f"\nSample feature names:")
print([f for f in FEATURES if 'trend' in f or 'roc' in f][:8])

Rows after dropping NaN lag rows: 218,700
Total features: 150
  Time features:  6
  Lag features:   144

Sample feature names:
['NO2_trend_3h', 'NO2_trend_6h', 'PM2.5_trend_3h', 'PM2.5_trend_6h', 'PM10_trend_3h', 'PM10_trend_6h', 'O3_trend_3h', 'O3_trend_6h']


Train/test split and save

In [7]:
import os

BASE_OUT = "C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk"
os.makedirs(f"{BASE_OUT}/models", exist_ok=True)

train = data[data['year'] <= 2024].copy()
test  = data[data['year'] == 2025].copy()

print(f"Train: {len(train):,} rows  "
      f"({train['datetime'].min().date()} → {train['datetime'].max().date()})")
print(f"Test:  {len(test):,} rows   "
      f"({test['datetime'].min().date()} → {test['datetime'].max().date()})")

print(f"\nTrain 6h target:")
for i, label in enumerate(['Low','Moderate','Dangerous']):
    n = (train['DAQI_6h']==i).sum()
    print(f"  {i} {label:10s}: {n:6,}  ({n/len(train)*100:.1f}%)")

print(f"\nTrain 12h target:")
for i, label in enumerate(['Low','Moderate','Dangerous']):
    n = (train['DAQI_12h']==i).sum()
    print(f"  {i} {label:10s}: {n:6,}  ({n/len(train)*100:.1f}%)")

# Save combined dataset (has both targets)
train.to_csv(f"{BASE_OUT}/models/train_forecast.csv", index=False)
test.to_csv(f"{BASE_OUT}/models/test_forecast.csv",  index=False)

# Save feature lists
pd.Series(FEATURES).to_csv(
    f"{BASE_OUT}/models/feature_list.csv", index=False)
pd.Series(FEATURES).to_csv(
    f"{BASE_OUT}/models/feature_list_12h.csv", index=False)

# Save medians for app
train[FEATURES].median().to_csv(f"{BASE_OUT}/models/train_median.csv")
train[FEATURES].median().to_csv(f"{BASE_OUT}/models/train_median_12h.csv")

print(f"\nSaved:")
print(f"  train_forecast.csv  — {len(train):,} rows")
print(f"  test_forecast.csv   — {len(test):,} rows")
print(f"  feature_list.csv    — {len(FEATURES)} features")
print(f"  feature_list_12h.csv — {len(FEATURES)} features")

Train: 174,955 rows  (2021-01-04 → 2024-12-31)
Test:  43,745 rows   (2025-01-01 → 2025-12-31)

Train 6h target:
  0 Low       : 171,543  (98.0%)
  1 Moderate  :  3,007  (1.7%)
  2 Dangerous :    405  (0.2%)

Train 12h target:
  0 Low       : 171,543  (98.0%)
  1 Moderate  :  3,007  (1.7%)
  2 Dangerous :    405  (0.2%)

Saved:
  train_forecast.csv  — 174,955 rows
  test_forecast.csv   — 43,745 rows
  feature_list.csv    — 150 features
  feature_list_12h.csv — 150 features
